# Lesson 4 — CRUD · เพิ่ม แก้ ลบ upsert

LanceDB ใช้เป็น database ธรรมดาได้ ไม่ต้องมี vector ก็ได้
บทนี้ทำครบ create · update · delete · upsert แล้วดูว่า disk เปลี่ยนยังไง
กฎเดิม ไม่มีอะไรถูกเขียนทับ มีแต่เขียนเพิ่ม

In [1]:
%pip install -q lancedb pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import lancedb
from pathlib import Path

def tree(root: Path, prefix: str = ""):
    kids = sorted(root.iterdir(), key=lambda p: (p.is_file(), p.name))
    for i, p in enumerate(kids):
        last = i == len(kids) - 1
        print(prefix + ("└── " if last else "├── ") + p.name)
        if p.is_dir():
            tree(p, prefix + ("    " if last else "│   "))

db = lancedb.connect("./data")

**Create** ตารางไม่มี vector เลย column ธรรมดาสามอัน

In [3]:
tbl = db.create_table("users", data=[
    {"id": 1, "name": "nat",  "plan": "free"},
    {"id": 2, "name": "beta", "plan": "free"},
    {"id": 3, "name": "thor", "plan": "pro"},
], mode="overwrite")
tbl.to_pandas()

[2026-09-10T06:34:38Z WARN  lance::dataset::write::insert] No existing dataset at /opt/Code/github.com/Soul-Brews-Studio/lancedb-oracle/lessons/04-crud/data/users.lance, it will be created


,id,name,plan
0,1,nat,free
1,2,beta,free
2,3,thor,pro


**Update** ใช้ `where` เลือกแถว `values` บอกค่าใหม่
แถวที่โดนแก้ไม่ได้ถูกแก้ในที่ Lance เขียนแถวใหม่ แล้ว mark แถวเก่าว่าลบ

In [4]:
tbl.update(where="id = 2", values={"plan": "pro"})
tbl.to_pandas()

,id,name,plan
0,1,nat,free
1,3,thor,pro
2,2,beta,pro


**Delete** ก็เหมือนกัน ไฟล์ data ไม่ถูกแตะ
มีไฟล์ใหม่โผล่ใน `_deletions/` บอกว่าแถวไหนไม่ต้องอ่านแล้ว

In [5]:
tbl.delete("id = 3")
tbl.to_pandas()

,id,name,plan
0,1,nat,free
1,2,beta,pro


**Upsert** ด้วย `merge_insert` เจอ `id` ซ้ำก็ update ไม่เจอก็ insert
คำสั่งเดียว ทำสองอย่าง

In [6]:
(
    tbl.merge_insert("id")
    .when_matched_update_all()
    .when_not_matched_insert_all()
    .execute([
        {"id": 1, "name": "nat", "plan": "team"},  # exists -> update
        {"id": 4, "name": "odin", "plan": "free"},  # new -> insert
    ])
)
tbl.to_pandas()

,id,name,plan
0,2,beta,pro
1,1,nat,team
2,4,odin,free


เปิด disk ดู 4 คำสั่ง 4 version
`_deletions/` คือร่องรอยของ update กับ delete
แถวที่ "หาย" จากตาราง ยังอยู่ในไฟล์ data เดิมทั้งหมด

In [7]:
print("version:", tbl.version)
tree(Path("data/users.lance"))

version: 4
├── _deletions
│   ├── 0-1-10350897025592884348.arrow
│   └── 0-2-14905981262751587764.arrow
├── _transactions
│   ├── 0-cb95ee17-485c-4efb-b27c-b6bdd5bcc573.txn
│   ├── 1-06f8a589-54ac-4d35-9d50-b13f96703c4d.txn
│   ├── 2-9e7d97dc-afeb-4ab9-a0be-0645420f4004.txn
│   └── 3-709a77d3-9ec4-4ebe-b58c-61c9ec865a68.txn
├── _versions
│   ├── 18446744073709551611.manifest
│   ├── 18446744073709551612.manifest
│   ├── 18446744073709551613.manifest
│   ├── 18446744073709551614.manifest
│   └── latest_version_hint.json
└── data
    ├── 0100111010110000110010005c3bbd4964ac0a9123a55aa6c7.lance
    ├── 0110101101111101001010112026e8479ca1eb8bef595b7db7.lance
    └── 110101011110110100010001f142464ce5b851303eb10eff37.lance
